# Notebook 3: DL Strategy (LSTM)
**Purpose:** Walk-forward LSTM training to generate BUY/SELL/HOLD signals for all 20 assets.  
**Prerequisites:** Run `0_data_and_features.ipynb` first.  
**GPU:** Auto-selected via pynvml. Falls back to CPU per-asset on OOM.  
**Runtime:** ~60-90 min on GPU, longer on CPU.

In [1]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pynvml

from config import (
    ALL_TICKERS, FEATURES_DIR, CAPITAL_PER_ASSET, FEATURE_COLS,
)
from utils import (
    setup_logging, create_directories, load_exchange_rates,
    simulate_trading, compute_metrics, save_results, is_indian_stock,
)

logger = setup_logging('dl_strategy')
create_directories()

# DL hyperparameters — defined here since they're DL-specific
DL_TRAIN_WINDOW = 500   # ~2 years of trading days
DL_RETRAIN_FREQ = 60    # retrain every ~1 quarter
SEQ_LEN = 60            # 60-day lookback per sequence
BATCH_SIZE = 32
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 10
LEARNING_RATE = 0.001
LR_PATIENCE = 5
LR_FACTOR = 0.5
MIN_FREE_VRAM_GB = 4.0  # minimum VRAM to attempt GPU training

# load common backtest start date (set in notebook 0)
with open(os.path.join('data', 'common_start_date.json'), 'r') as f:
    common_start = json.load(f)['common_start_date']
print(f'Common backtest start: {common_start}')

exchange_rates = load_exchange_rates()
print('Setup complete.')

Common backtest start: 2022-04-30
Setup complete.


## 1. GPU Selection

In [2]:
def select_gpu(min_free_gb=MIN_FREE_VRAM_GB):
    """
    On shared DGX, free VRAM varies across GPUs depending on other users.
    Pick the GPU with the most free VRAM right now.
    Falls back to CPU if nothing has enough free memory.
    """
    try:
        pynvml.nvmlInit()
        n_gpus = pynvml.nvmlDeviceGetCount()
        free_mem_gb = []

        for i in range(n_gpus):
            handle = pynvml.nvmlDeviceGetHandleByIndex(i)
            info = pynvml.nvmlDeviceGetMemoryInfo(handle)
            free_gb = info.free / (1024 ** 3)
            free_mem_gb.append(free_gb)
            print(f'  GPU {i}: {free_gb:.1f} GB free / {info.total / (1024**3):.1f} GB total')

        pynvml.nvmlShutdown()

        best = max(range(n_gpus), key=lambda i: free_mem_gb[i])

        if free_mem_gb[best] < min_free_gb:
            logger.warning(f'No GPU has >= {min_free_gb}GB free VRAM. Using CPU.')
            print(f'WARNING: All GPUs below {min_free_gb}GB free — falling back to CPU.')
            return 'cpu'

        # tell PyTorch to only see this GPU (avoids accidental multi-GPU usage)
        os.environ['CUDA_VISIBLE_DEVICES'] = str(best)
        print(f'=> Selected GPU {best} ({free_mem_gb[best]:.1f} GB free)')
        return 'cuda'

    except Exception as e:
        logger.warning(f'pynvml error: {e}. Defaulting to CPU.')
        print(f'GPU check failed ({e}), using CPU.')
        return 'cpu'


print('Checking available GPUs...')
DEVICE = select_gpu()
print(f'Training device: {DEVICE}')

Checking available GPUs...
  GPU 0: 0.1 GB free / 32.0 GB total
  GPU 1: 7.4 GB free / 32.0 GB total
  GPU 2: 17.9 GB free / 32.0 GB total
  GPU 3: 25.7 GB free / 32.0 GB total
  GPU 4: 31.3 GB free / 32.0 GB total
  GPU 5: 24.6 GB free / 32.0 GB total
  GPU 6: 18.4 GB free / 32.0 GB total
  GPU 7: 28.4 GB free / 32.0 GB total
=> Selected GPU 4 (31.3 GB free)
Training device: cuda


## 2. LSTM Model

In [3]:
class LSTMClassifier(nn.Module):
    """
    2-layer LSTM that classifies each 60-day price sequence as SELL / HOLD / BUY.

    Architecture (per spec):
        Input (batch, seq_len=60, n_features)
        -> LSTM(128) -> Dropout(0.3)
        -> LSTM(64)  -> Dropout(0.3)
        -> Linear(3)   [raw logits, CrossEntropyLoss handles softmax]
    """
    def __init__(self, n_features, hidden1=128, hidden2=64, dropout=0.3):
        super().__init__()
        self.lstm1 = nn.LSTM(n_features, hidden1, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(hidden1, hidden2, batch_first=True)
        self.drop2 = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden2, 3)

    def forward(self, x):
        # x: (batch, seq_len, features)
        out, _ = self.lstm1(x)
        out = self.drop1(out)
        out, _ = self.lstm2(out)
        out = self.drop2(out)
        # only use the last timestep's hidden state for classification
        out = self.fc(out[:, -1, :])
        return out  # raw logits


class SequenceDataset(Dataset):
    """Simple wrapper so we can use DataLoader."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## 3. Sequence Creation & Normalization

In [4]:
def normalize_sequence(seq):
    """
    Normalize a single (seq_len, n_features) array using its own stats.
    Uses RobustScaler logic: (x - median) / IQR per feature.
    
    Doing this per-sequence avoids leaking scale info from training into test.
    At inference time you only have the last 60 days anyway, so this is realistic.
    """
    median = np.median(seq, axis=0)
    q75 = np.percentile(seq, 75, axis=0)
    q25 = np.percentile(seq, 25, axis=0)
    iqr = (q75 - q25) + 1e-8  # small epsilon to avoid div-by-zero on flat features
    return (seq - median) / iqr


def build_sequences(X, y, seq_len=SEQ_LEN):
    """
    Convert flat (n_days, n_features) arrays into overlapping 60-day sequences.
    
    For day i, the sequence is X[i-seq_len : i] and label is y[i].
    Each sequence is normalized independently before being added to the batch.
    
    Returns:
        X_seq: (n_samples, seq_len, n_features) float32
        y_seq: (n_samples,) int64
    """
    X_seqs, y_seqs = [], []

    for i in range(seq_len, len(X)):
        seq = X[i - seq_len : i].copy()
        seq = normalize_sequence(seq)
        X_seqs.append(seq)
        y_seqs.append(y[i])

    return (
        np.array(X_seqs, dtype=np.float32),
        np.array(y_seqs, dtype=np.int64),
    )

## 4. Single-Model Training

In [5]:
def train_lstm(X_train, y_train, n_features, device, asset_num, n_assets, cycle_num, n_cycles):
    """
    Train one LSTM model on the given training window.
    
    - Validation = last 20% of training data (time-ordered, no shuffle)
    - Early stopping on validation loss (patience=10)
    - ReduceLROnPlateau scheduler
    - If GPU OOM, retries automatically on CPU for this asset
    
    Returns: (trained model, device_used) or (None, device) if training fails
    """
    X_seq, y_seq = build_sequences(X_train, y_train)

    if len(X_seq) < 10:
        logger.warning(f'Too few sequences ({len(X_seq)}) to train — skipping cycle.')
        return None, device

    # time-ordered val split: last 20%
    val_size = max(1, int(len(X_seq) * 0.2))
    X_tr, X_val = X_seq[:-val_size], X_seq[-val_size:]
    y_tr, y_val = y_seq[:-val_size], y_seq[-val_size:]

    train_loader = DataLoader(SequenceDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=False)
    val_loader   = DataLoader(SequenceDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)

    # try training on the requested device; fall back to CPU on OOM
    devices_to_try = [device, 'cpu'] if device != 'cpu' else ['cpu']

    for dev in devices_to_try:
        try:
            model     = LSTMClassifier(n_features).to(dev)
            optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE, verbose=False
            )
            criterion = nn.CrossEntropyLoss()

            best_val_loss   = float('inf')
            patience_count  = 0
            best_weights    = None

            for epoch in range(MAX_EPOCHS):
                # train pass
                model.train()
                for X_batch, y_batch in train_loader:
                    X_batch, y_batch = X_batch.to(dev), y_batch.to(dev)
                    optimizer.zero_grad()
                    loss = criterion(model(X_batch), y_batch)
                    loss.backward()
                    optimizer.step()

                # validation pass
                model.eval()
                val_loss = 0.0
                with torch.no_grad():
                    for X_batch, y_batch in val_loader:
                        X_batch, y_batch = X_batch.to(dev), y_batch.to(dev)
                        val_loss += criterion(model(X_batch), y_batch).item()
                val_loss /= len(val_loader)

                scheduler.step(val_loss)

                # progress line — overwrites itself each epoch
                print(
                    f'\r  Asset [{asset_num}/{n_assets}] '
                    f'Cycle [{cycle_num}/{n_cycles}] '
                    f'Epoch [{epoch+1:>2}/{MAX_EPOCHS}] '
                    f'val_loss={val_loss:.4f}  ',
                    end='', flush=True
                )

                # save best weights & check early stopping
                if val_loss < best_val_loss:
                    best_val_loss  = val_loss
                    # store on CPU so we don't hold extra VRAM
                    best_weights   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    patience_count = 0
                else:
                    patience_count += 1

                if patience_count >= EARLY_STOP_PATIENCE:
                    break

            print()  # newline after the epoch progress bar

            # restore best checkpoint
            if best_weights is not None:
                model.load_state_dict({k: v.to(dev) for k, v in best_weights.items()})

            return model, dev

        except torch.cuda.OutOfMemoryError:
            print()
            logger.warning(f'OOM on GPU — retrying on CPU for this asset.')
            torch.cuda.empty_cache()
            # loop will retry with 'cpu'

        except Exception as e:
            print()
            logger.error(f'Training error at cycle {cycle_num}: {e}')
            return None, dev

    return None, 'cpu'

## 5. Walk-Forward Inference

In [6]:
def walk_forward_dl(df, device, asset_num, n_assets, ticker):
    """
    Walk-forward training + inference for a single asset.
    
    Flow per retrain cycle:
        1. Grab the last DL_TRAIN_WINDOW valid rows as training data
        2. Train a fresh LSTM on those rows
        3. Run inference on the next DL_RETRAIN_FREQ rows
        4. Move the window forward by DL_RETRAIN_FREQ and repeat
    
    Signals are shifted by 1 day at the end (signal on day T -> trade on open of T+1).
    Default signal for all days is HOLD (covers warm-up period & failed cycles).
    
    Returns: pd.Series of signals ('BUY'/'SELL'/'HOLD') indexed by df.index
    """
    # drop rows where features or target are NaN (warm-up / end of series)
    valid_df = df[FEATURE_COLS + ['target']].dropna()

    min_rows_needed = DL_TRAIN_WINDOW + DL_RETRAIN_FREQ + SEQ_LEN
    if len(valid_df) < min_rows_needed:
        logger.warning(f'{ticker}: not enough data ({len(valid_df)} rows, need {min_rows_needed}). Returning HOLD.')
        return pd.Series('HOLD', index=df.index)

    dates   = valid_df.index
    X_all   = valid_df[FEATURE_COLS].values         # (n_valid_days, 15)
    y_all   = valid_df['target'].values.astype(int)  # 0=SELL 1=HOLD 2=BUY
    n_feat  = X_all.shape[1]
    sig_map = {0: 'SELL', 1: 'HOLD', 2: 'BUY'}

    # how many retrain cycles will we run? (for progress display)
    n_cycles = max(1, (len(valid_df) - DL_TRAIN_WINDOW) // DL_RETRAIN_FREQ)

    all_signals     = pd.Series('HOLD', index=df.index)
    current_device  = device
    model           = None
    last_train_idx  = -1
    cycle_num       = 0
    start_idx       = DL_TRAIN_WINDOW  # first position we can make a prediction

    while start_idx < len(valid_df):
        train_end  = start_idx
        train_start = max(0, train_end - DL_TRAIN_WINDOW)
        test_end   = min(len(valid_df), start_idx + DL_RETRAIN_FREQ)

        # retrain when we've moved at least DL_RETRAIN_FREQ steps since last training
        needs_retrain = (model is None) or (start_idx - last_train_idx >= DL_RETRAIN_FREQ)

        if needs_retrain:
            cycle_num += 1
            X_train = X_all[train_start:train_end]
            y_train = y_all[train_start:train_end]

            result, current_device = train_lstm(
                X_train, y_train, n_feat, current_device,
                asset_num, n_assets, cycle_num, n_cycles
            )

            if result is None:
                # training failed for this cycle, stay on HOLD and move on
                logger.warning(f'{ticker}: cycle {cycle_num} training failed — using HOLD.')
                start_idx = test_end
                continue

            model          = result
            last_train_idx = start_idx

        # ── inference on the test window ──────────────────────────────────
        model.eval()

        # collect all 60-day sequences for the test window in one batch
        seqs = []
        valid_test_positions = []  # which positions in the test window have enough history

        for j, abs_idx in enumerate(range(start_idx, test_end)):
            if abs_idx < SEQ_LEN:
                # not enough history yet — stays HOLD
                continue
            seq = X_all[abs_idx - SEQ_LEN : abs_idx].copy()
            seq = normalize_sequence(seq)
            seqs.append(seq)
            valid_test_positions.append(j)

        if seqs:
            batch = torch.tensor(np.array(seqs), dtype=torch.float32).to(current_device)
            with torch.no_grad():
                logits = model(batch)                        # (n_test, 3)
                preds  = logits.argmax(dim=1).cpu().numpy()  # (n_test,)

            test_dates = dates[start_idx:test_end]
            for j, pred in zip(valid_test_positions, preds):
                all_signals.loc[test_dates[j]] = sig_map.get(int(pred), 'HOLD')

        start_idx = test_end

    # free up VRAM for the next asset
    if current_device != 'cpu':
        torch.cuda.empty_cache()

    # shift by 1 day: signal generated at close of T -> trade executes at open of T+1
    all_signals = all_signals.shift(1).fillna('HOLD')

    return all_signals

## 6. Run LSTM Strategy — All Assets

In [7]:
# load feature data
def load_features(ticker):
    safe_name = ticker.replace('=', '_').replace('-', '_')
    path = os.path.join(FEATURES_DIR, f'{safe_name}_features.parquet')
    return pd.read_parquet(path)

feature_data = {}
for ticker in ALL_TICKERS:
    feature_data[ticker] = load_features(ticker)
print(f'Loaded features for {len(feature_data)} assets.')

Loaded features for 20 assets.


In [8]:
lstm_portfolios = {}
lstm_orderbooks = {}
lstm_metrics    = {}

n_assets = len(feature_data)

for i, (ticker, df) in enumerate(feature_data.items()):
    print(f'\n[{i+1}/{n_assets}] LSTM — {ticker}')
    print('-' * 50)

    try:
        signals = walk_forward_dl(df, DEVICE, i + 1, n_assets, ticker)

        # Indian stocks need INR->USD conversion during simulation
        exr = exchange_rates if is_indian_stock(ticker) else None

        ph, ob = simulate_trading(
            df, signals, ticker,
            initial_capital=CAPITAL_PER_ASSET,
            exchange_rates=exr,
            start_date=common_start,
        )

        lstm_portfolios[ticker] = ph
        lstm_orderbooks[ticker] = ob

        m = compute_metrics(ph)
        lstm_metrics[ticker] = m

        print(f'  Done — Return: {m["total_return"]*100:+.1f}% | '
              f'Sharpe: {m["sharpe_ratio"]:.2f} | '
              f'MaxDD: {m["max_drawdown"]*100:.1f}% | '
              f'Trades: {m["num_trades"]}')

    except Exception as e:
        print(f'  FAILED: {e}')
        logger.error(f'LSTM failed for {ticker}: {e}')

print(f'\nLSTM strategy complete: {len(lstm_metrics)}/{n_assets} assets.')


[1/20] LSTM — AAPL
--------------------------------------------------
  Asset [1/20] Cycle [1/27] Epoch [13/50] val_loss=1.1076  
  Asset [1/20] Cycle [2/27] Epoch [15/50] val_loss=1.0703  
  Asset [1/20] Cycle [3/27] Epoch [11/50] val_loss=1.3376  
  Asset [1/20] Cycle [4/27] Epoch [11/50] val_loss=1.4203  
  Asset [1/20] Cycle [5/27] Epoch [11/50] val_loss=1.2621  
  Asset [1/20] Cycle [6/27] Epoch [11/50] val_loss=1.4256  
  Asset [1/20] Cycle [7/27] Epoch [11/50] val_loss=1.2705  
  Asset [1/20] Cycle [8/27] Epoch [16/50] val_loss=1.4177  
  Asset [1/20] Cycle [9/27] Epoch [11/50] val_loss=1.3345  
  Asset [1/20] Cycle [10/27] Epoch [13/50] val_loss=1.1840  
  Asset [1/20] Cycle [11/27] Epoch [11/50] val_loss=1.2233  
  Asset [1/20] Cycle [12/27] Epoch [11/50] val_loss=1.1604  
  Asset [1/20] Cycle [13/27] Epoch [11/50] val_loss=1.2123  
  Asset [1/20] Cycle [14/27] Epoch [19/50] val_loss=1.3221  
  Asset [1/20] Cycle [15/27] Epoch [11/50] val_loss=1.2698  
  Asset [1/20] Cycle [1

## 7. Save Results & Summary

In [9]:
# save portfolio histories and order books to results/
save_results(lstm_portfolios, lstm_orderbooks, 'dl_lstm')
print('Results saved to results/ directory.')

# print summary table
if lstm_metrics:
    print(f'\n{"Ticker":<20} {"Return":>8} {"Sharpe":>8} {"MaxDD":>8} {"Trades":>8}')
    print('-' * 58)
    for ticker, m in lstm_metrics.items():
        print(
            f'{ticker:<20} '
            f'{m["total_return"]*100:>+7.1f}% '
            f'{m["sharpe_ratio"]:>8.2f} '
            f'{m["max_drawdown"]*100:>7.1f}% '
            f'{m["num_trades"]:>8}'
        )

    avg_ret    = np.mean([m['total_return']  for m in lstm_metrics.values()])
    avg_sharpe = np.mean([m['sharpe_ratio']  for m in lstm_metrics.values()])
    avg_dd     = np.mean([m['max_drawdown']  for m in lstm_metrics.values()])
    print('-' * 58)
    print(f'  Average: return={avg_ret*100:+.1f}%  sharpe={avg_sharpe:.2f}  maxDD={avg_dd*100:.1f}%')

print('\nNotebook 3 complete.')

Results saved to results/ directory.

Ticker                 Return   Sharpe    MaxDD   Trades
----------------------------------------------------------
AAPL                   +21.6%     0.71     9.1%      703
MSFT                   +21.8%     0.53    12.7%      759
GOOGL                  +10.5%     0.34    13.4%      893
AMZN                   +14.8%     0.48    12.1%      904
NVDA                   +46.6%     0.91    12.2%      969
META                   +27.9%     0.78     7.8%      952
TSLA                   +16.2%     0.57     8.4%      972
JPM                    +17.9%     0.69     7.8%      555
JNJ                     -6.1%    -0.22    10.9%        9
V                      +23.2%     1.07     6.1%      451
RELIANCE.NS            +21.3%     0.63    13.9%      647
TCS.NS                  +9.6%     0.33    10.3%      278
HDFCBANK.NS             +6.8%     0.27    13.6%      349
INFY.NS                +18.1%     0.48    14.8%      530
ICICIBANK.NS           +15.7%     0.46     8.0% 